# Auditoria de qualidade do bronze: cadastro e funil comercial

Notebook de auditoria da **Fictoria Casa & Interiores** (empresa fictícia, dados 100% sintéticos). Lê o **bronze** (estado corrente por chave) com DuckDB e evidencia, achado a achado, a sujeira que a **silver** precisa tratar. Cada seção termina numa **Nota Técnica** (Observado · Por que importa · Ação) escrita a partir do que a célula mostrou; o conjunto das ações é o [catálogo de achados](../docs/08_catalogo_achados_silver.md), o contrato da silver.

> Reprodutível: `uv run notebooks-qualidade` reconstrói e reexecuta este notebook a partir de `src/interiores_fictoria/qualidade/achados.py`.

In [1]:
import duckdb, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from interiores_fictoria import duck

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
con, lake = duck.conectar()
print("lake:", lake.raiz)
print("cargas no bronze:", len(lake.listar("bronze", "_controle", "cargas")))


lake: /home/neviah/projetos/interiores-fictoria-bigdata/data/lake
cargas no bronze: 5


## 0. Retrato do funil

Antes dos defeitos, o cenário: volumes, desfechos e o arco da conversão por ano.

In [2]:
# Retrato do funil: volumes, conversão e carteira por ano (o cenário que os achados afetam)
df = con.execute("""
SELECT YEAR(o.dt_cadastro) AS ano, COUNT(*) AS orcamentos,
       SUM(CASE WHEN f.grupo = 'GANHO' THEN 1 ELSE 0 END) AS ganhos,
       SUM(CASE WHEN f.grupo = 'PERDIDO' THEN 1 ELSE 0 END) AS perdidos,
       SUM(CASE WHEN f.grupo = 'ABERTO' THEN 1 ELSE 0 END) AS abertos,
       ROUND(SUM(CASE WHEN f.grupo = 'GANHO' THEN 1.0 ELSE 0 END)
             / NULLIF(SUM(CASE WHEN f.grupo IN ('GANHO','PERDIDO') THEN 1.0 ELSE 0 END), 0), 3)
           AS conversao_sobre_fechados,
       ROUND(SUM(o.vl_total_liquido) / 1e6, 1) AS liquido_mi
FROM b_orcamento o JOIN b_fase f ON f.id = o.fase_id
WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL
GROUP BY 1 ORDER BY 1
""").df()
display(df)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(df["ano"].astype(str), df["orcamentos"], color="#c9b79c", label="orçamentos")
ax2 = ax.twinx()
ax2.plot(df["ano"].astype(str), df["conversao_sobre_fechados"] * 100, color="#5b4a2f", marker="o",
         label="conversão sobre fechados (%)")
ax.set_title("Orçamentos por ano e o arco da conversão")
ax2.set_ylim(0, 60)
for spine in ("top",):
    ax.spines[spine].set_visible(False); ax2.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


,ano,orcamentos,ganhos,perdidos,abertos,conversao_sobre_fechados,liquido_mi
0,2021,1486,132.0,1354.0,0.0,0.089,168.9
1,2022,1719,254.0,1465.0,0.0,0.148,201.7
2,2023,2333,432.0,1901.0,0.0,0.185,316.0
3,2024,2571,639.0,1932.0,0.0,0.249,343.9
4,2025,3517,1079.0,2354.0,84.0,0.314,516.5
5,2026,2554,868.0,877.0,809.0,0.497,400.9


/tmp/ipykernel_107187/4096069952.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1. ACH-01 · Data-sentinela 1900-01-01 no fechamento

In [3]:
df = con.execute("""SELECT fase_id, COUNT(*) AS orcamentos,
                   SUM(CASE WHEN dt_finalizou = TIMESTAMP '1900-01-01' THEN 1 ELSE 0 END) AS sentinela
            FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND fase_id IN (6, 7) GROUP BY fase_id ORDER BY fase_id""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN dt_finalizou = TIMESTAMP '1900-01-01' THEN 1.0 ELSE 0.0 END)
            FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND fase_id IN (6, 7)""").fetchone()[0]
print("métrica ACH-01:", metrica)

,fase_id,orcamentos,sentinela
0,6,3404,50.0
1,7,9883,209.0


métrica ACH-01: 0.019492737261985398


**Nota Técnica ACH-01**

- **Observado:** taxa observada de 1,95% na base analisada (métrica: `ACH-01`).
- **Por que importa:** Um orçamento fechado com data 1900 entra em qualquer filtro de período como se fosse do século passado e some dos relatórios do ano; somado a datas nulas, distorce prazos médios.
- **Ação (regra da silver):** `dt_finalizou` sentinela vira NULL e a data de fechamento passa a ser a entrada na fase final registrada na trilha (`orcamento_fase_hist`), que não tem sentinela; flag `fl_dt_finalizou_sentinela`.

## 2. ACH-02 · Fechamento anterior ao cadastro

In [4]:
df = con.execute("""SELECT id, nr_orcamento, dt_cadastro, dt_finalizou,
                   date_diff('day', dt_cadastro, dt_finalizou) AS dias
            FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND fase_id IN (6, 7) AND dt_finalizou > TIMESTAMP '1900-01-01'
              AND dt_finalizou < dt_cadastro ORDER BY dias LIMIT 15""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN dt_finalizou > TIMESTAMP '1900-01-01' AND dt_finalizou < dt_cadastro
                            THEN 1.0 ELSE 0.0 END) FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND fase_id IN (6, 7)""").fetchone()[0]
print("métrica ACH-02:", metrica)

,id,nr_orcamento,dt_cadastro,dt_finalizou,dias
0,9183,10183,2025-02-25 11:39:00,2023-07-06 11:39:00,-600
1,7143,8143,2024-06-17 15:40:17,2022-10-27 15:40:17,-599
2,14869,15869,2026-08-05 15:25:04,2025-03-11 15:25:04,-512
3,7787,8787,2024-09-09 15:10:54,2023-05-08 15:10:54,-490
4,11953,12953,2025-11-06 09:33:25,2024-07-04 09:33:25,-490
5,11165,12165,2025-09-01 16:42:18,2024-05-27 16:42:18,-462
6,12353,13353,2025-12-10 14:30:06,2024-10-17 14:30:06,-419
7,11696,12696,2025-10-15 10:56:33,2024-09-07 10:56:33,-403
8,53,1053,2021-01-14 14:05:25,2019-12-22 14:05:25,-389
9,9743,10743,2025-04-19 16:59:08,2024-04-07 16:59:08,-377


métrica ACH-02: 0.001956799879581546


**Nota Técnica ACH-02**

- **Observado:** taxa observada de 0,20% na base analisada (métrica: `ACH-02`).
- **Por que importa:** Dias até fechar negativos derrubam a média e a mediana de prazo; a causa típica é relógio de estação errado ou digitação manual da data de fechamento.
- **Ação (regra da silver):** A data de fechamento confiável vem da trilha de fases; o valor registrado fica preservado em `dt_finalizou_registrado` com a flag `fl_fechou_antes_cadastro`.

## 3. ACH-03 · Orçamentos de valor zero

In [5]:
df = con.execute("""SELECT YEAR(dt_cadastro) AS ano, COUNT(*) AS orcamentos,
                   SUM(CASE WHEN vl_total_bruto = 0 THEN 1 ELSE 0 END) AS valor_zero
            FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL GROUP BY 1 ORDER BY 1""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN vl_total_bruto = 0 THEN 1.0 ELSE 0.0 END) FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL""").fetchone()[0]
print("métrica ACH-03:", metrica)

,ano,orcamentos,valor_zero
0,2021,1486,32.0
1,2022,1719,22.0
2,2023,2333,36.0
3,2024,2571,33.0
4,2025,3517,43.0
5,2026,2554,26.0


métrica ACH-03: 0.013540197461212976


**Nota Técnica ACH-03**

- **Observado:** taxa observada de 1,35% na base analisada (métrica: `ACH-03`).
- **Por que importa:** Valor zero conta como orçamento no volume e não conta em dinheiro: ticket médio e conversão em valor ficam subestimados sem que ninguém perceba.
- **Ação (regra da silver):** Mantido no funil (é um orçamento real) com `fl_valor_zero`; a gold exclui esses orçamentos das médias de ticket e os mostra como indicador próprio.

## 4. ACH-04 · Total bruto diferente da soma dos itens

In [6]:
df = con.execute("""SELECT YEAR(o.dt_cadastro) AS ano, COUNT(*) AS orcamentos,
                  SUM(CASE WHEN ABS(o.vl_total_bruto - COALESCE(i.soma, 0)) > 0.05 THEN 1 ELSE 0 END)
                      AS divergentes,
                  SUM(CASE WHEN ABS(o.vl_total_bruto - COALESCE(i.soma_com_cancelados, 0)) <= 0.05
                            AND ABS(o.vl_total_bruto - COALESCE(i.soma, 0)) > 0.05 THEN 1 ELSE 0 END)
                      AS cancelado_somado
           FROM b_orcamento o
           LEFT JOIN (SELECT orcamento_id,
                             SUM(CASE WHEN dt_cancelou IS NULL THEN vl_total ELSE 0 END) AS soma,
                             SUM(vl_total) AS soma_com_cancelados
                      FROM b_orcamento_item GROUP BY 1) i ON i.orcamento_id = o.id
           WHERE o.dt_cancelou IS NULL GROUP BY 1 ORDER BY 1""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN ABS(o.vl_total_bruto - COALESCE(i.soma, 0)) > 0.05 THEN 1.0 ELSE 0.0 END)
           FROM b_orcamento o
           LEFT JOIN (SELECT orcamento_id, SUM(vl_total) AS soma FROM b_orcamento_item
                      WHERE dt_cancelou IS NULL GROUP BY 1) i ON i.orcamento_id = o.id
           WHERE o.dt_cancelou IS NULL""").fetchone()[0]
print("métrica ACH-04:", metrica)

,ano,orcamentos,divergentes,cancelado_somado
0,2021,1577,8.0,8.0
1,2022,1834,8.0,8.0
2,2023,2487,0.0,0.0
3,2024,2724,0.0,0.0
4,2025,3742,0.0,0.0
5,2026,2704,11.0,0.0


métrica ACH-04: 0.0017918768250597293


**Nota Técnica ACH-04**

- **Observado:** taxa observada de 0,18% na base analisada (métrica: `ACH-04`).
- **Por que importa:** O cabeçalho é o que a empresa vê; os itens são o que ela vendeu. Quando divergem, o relatório por categoria de item não fecha com o total do orçamento. Aqui a divergência é um bug antigo do sistema (2021-2022): item cancelado continuou somado.
- **Ação (regra da silver):** A silver recalcula `vl_bruto_itens` a partir dos itens não cancelados e marca `fl_divergencia_itens`; a gold usa o cabeçalho (o que foi negociado) e expõe a divergência.

## 5. ACH-05 · Clientes duplicados por grafia

In [7]:
df = con.execute("""WITH n AS (
             SELECT id, nome,
                    regexp_replace(strip_accents(lower(trim(nome))), '\s+', ' ', 'g') AS nome_norm
             FROM b_cliente)
           SELECT nome_norm, COUNT(*) AS cadastros, string_agg(nome, ' | ' ORDER BY id) AS grafias
           FROM n GROUP BY 1 HAVING COUNT(*) > 1 ORDER BY 2 DESC, 1 LIMIT 15""").df()
display(df)
metrica = con.execute("""WITH n AS (
             SELECT regexp_replace(strip_accents(lower(trim(nome))), '\s+', ' ', 'g') AS nome_norm
             FROM b_cliente)
           SELECT 1.0 - COUNT(DISTINCT nome_norm) * 1.0 / COUNT(*) FROM n""").fetchone()[0]
print("métrica ACH-05:", metrica)

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_107187/3802431086.py:1: SyntaxWarning: invalid escape sequence '\s'
  df = con.execute("""WITH n AS (
/tmp/ipykernel_107187/3802431086.py:8: SyntaxWarning: invalid escape sequence '\s'
  metrica = con.execute("""WITH n AS (


,nome_norm,cadastros,grafias
0,freitas consultoria ltda,10,Freitas Consultoria Ltda | Freitas Consultoria...
1,monteiro comercio ltda,10,Monteiro Comércio Ltda | Monteiro Comércio Ltd...
2,nascimento consultoria ltda,10,Nascimento Consultoria Ltda | Nascimento Consu...
3,azevedo comercio ltda,9,Azevedo Comércio Ltda | Azevedo Comércio Ltda ...
4,azevedo consultoria ltda,9,Azevedo Consultoria Ltda | Azevedo Consultoria...
5,barros comercio ltda,9,Barros Comércio Ltda | Barros Comércio Ltda | ...
6,cunha empreendimentos ltda,9,Cunha Empreendimentos Ltda | Cunha Empreendime...
7,cunha participacoes ltda,9,Cunha Participações Ltda | Cunha Participações...
8,rocha comercio ltda,9,Rocha Comércio Ltda | Rocha Comércio Ltda | Ro...
9,teixeira participacoes ltda,9,Teixeira Participações Ltda | Teixeira Partic...


métrica ACH-05: 0.11459103986847519


**Nota Técnica ACH-05**

- **Observado:** taxa observada de 11,46% na base analisada (métrica: `ACH-05`).
- **Por que importa:** O mesmo cliente com três grafias vira três clientes: recorrência subestimada, ranking de clientes fiéis errado e premiação indo para a pessoa errada.
- **Ação (regra da silver):** `nome_normalizado` (minúsculas, sem acento, espaços colapsados) e `cliente_canonico_id` (o menor id do grupo); a dimensão de cliente da gold usa o canônico.

## 6. ACH-06 · Endereços sem CEP ou sem bairro

In [8]:
df = con.execute("""SELECT tipo, COUNT(*) AS enderecos,
                  SUM(CASE WHEN cep IS NULL THEN 1 ELSE 0 END) AS sem_cep,
                  SUM(CASE WHEN bairro IS NULL THEN 1 ELSE 0 END) AS sem_bairro
           FROM b_endereco GROUP BY 1 ORDER BY 2 DESC""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN cep IS NULL OR bairro IS NULL THEN 1.0 ELSE 0.0 END) FROM b_endereco""").fetchone()[0]
print("métrica ACH-06:", metrica)

,tipo,enderecos,sem_cep,sem_bairro
0,RESIDENCIA,11201,852.0,525.0
1,CAMPO,1031,84.0,200.0
2,COMERCIAL,964,80.0,47.0
3,LITORAL,786,54.0,165.0


métrica ACH-06: 0.13810613646116435


**Nota Técnica ACH-06**

- **Observado:** taxa observada de 13,81% na base analisada (métrica: `ACH-06`).
- **Por que importa:** Bairro é a geografia do público-alvo (Jardim Europa para cima); sem ele, a leitura por região perde parte da base, e ausência não se preenche com chute.
- **Ação (regra da silver):** Ausência preservada como NULL; a dimensão de cliente usa o bairro do endereço principal e o membro 'Não informado' quando faltar.

## 7. ACH-07 · Parceiro informado em texto livre, sem cadastro

In [9]:
df = con.execute("""SELECT oc.codigo AS canal, COUNT(*) AS orcamentos,
                  SUM(CASE WHEN o.ds_objetivo LIKE 'Indicação:%' THEN 1 ELSE 0 END) AS com_texto_livre
           FROM b_orcamento o JOIN b_origem_contato oc ON oc.id = o.origem_contato_id
           WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL GROUP BY 1 ORDER BY 2 DESC""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN ds_objetivo LIKE 'Indicação:%' THEN 1.0 ELSE 0.0 END)
            FROM b_orcamento WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND parceiro_id IS NULL AND origem_contato_id = 9""").fetchone()[0]
print("métrica ACH-07:", metrica)

,canal,orcamentos,com_texto_livre
0,WHATSAPP,3687,0.0
1,IND_CLIENTE,3290,0.0
2,ARQUITETO,3092,0.0
3,SAC,1338,0.0
4,INSTAGRAM,1332,0.0
5,OUTROS,474,213.0
6,CONSTRUTORA,442,0.0
7,FACEBOOK,358,0.0
8,ANUNCIO,167,0.0


métrica ACH-07: 0.44936708860759494


**Nota Técnica ACH-07**

- **Observado:** taxa observada de 44,94% na base analisada (métrica: `ACH-07`).
- **Por que importa:** Indicações de arquiteto que não viraram cadastro caem em 'Outros' e o parceiro não recebe o crédito; é volume de parceria invisível para a premiação.
- **Ação (regra da silver):** A silver extrai `parceiro_texto_livre` do objetivo e mantém o canal OUTROS (regra do negócio: parceiro é PJ cadastrada); a gold mostra 'Outros / não cadastrado' como membro.

## 8. ACH-08 · Ganhos sem trilha de etapas de execução

In [10]:
df = con.execute("""SELECT YEAR(o.dt_cadastro) AS ano, COUNT(*) AS ganhos,
                  SUM(CASE WHEN e.orcamento_id IS NULL THEN 1 ELSE 0 END) AS sem_etapas
           FROM b_orcamento o
           LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_orcamento_etapa) e ON e.orcamento_id = o.id
           WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL AND o.fase_id = 6
           GROUP BY 1 ORDER BY 1""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN e.orcamento_id IS NULL THEN 1.0 ELSE 0.0 END)
            FROM b_orcamento o
            LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_orcamento_etapa) e ON e.orcamento_id = o.id
            WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL AND o.fase_id = 6""").fetchone()[0]
print("métrica ACH-08:", metrica)

,ano,ganhos,sem_etapas
0,2021,132,21.0
1,2022,254,57.0
2,2023,432,23.0
3,2024,639,34.0
4,2025,1079,80.0
5,2026,868,48.0


métrica ACH-08: 0.077262044653349


**Nota Técnica ACH-08**

- **Observado:** taxa observada de 7,73% na base analisada (métrica: `ACH-08`).
- **Por que importa:** Sem trilha, o status operacional é 'não definido' e a obra não aparece no acompanhamento; concentra-se nos ganhos antigos, antes do processo existir.
- **Ação (regra da silver):** Status operacional derivado: 'NÃO DEFINIDO' quando não há etapa; `fl_sem_trilha_execucao`.

## 9. ACH-09 · Follow-up ausente depende do vendedor

In [11]:
df = con.execute("""SELECT u.nome AS vendedor, COUNT(*) AS orcamentos,
                  ROUND(AVG(CASE WHEN f.orcamento_id IS NULL THEN 1.0 ELSE 0.0 END), 3) AS sem_follow_up
           FROM b_orcamento o JOIN b_usuario u ON u.id = o.vendedor_id
           LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_follow_up) f ON f.orcamento_id = o.id
           WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL
           GROUP BY 1 HAVING COUNT(*) >= 100 ORDER BY 3 DESC""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN f.orcamento_id IS NULL THEN 1.0 ELSE 0.0 END)
            FROM b_orcamento o
            LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_follow_up) f ON f.orcamento_id = o.id
            WHERE tipo_orcamento_id = 1 AND dt_cancelou IS NULL""").fetchone()[0]
print("métrica ACH-09:", metrica)

,vendedor,orcamentos,sem_follow_up
0,Helena Souza Toledo,604,0.758
1,Mariana Vieira Ferreira,800,0.753
2,Pedro Monteiro Moreira,391,0.737
3,Camila Barbosa Duarte,338,0.722
4,Bruno Almeida Moraes,278,0.687
5,Leonardo Pereira Carvalho,133,0.105
6,Daniel Garcia Bittencourt,405,0.096
7,Thiago Cardoso Pinto,864,0.074
8,Otávio Vieira Mendes,291,0.072
9,Gabriela Nogueira Bittencourt,568,0.063


métrica ACH-09: 0.16417489421720732


In [12]:
df = con.execute("""SELECT u.nome, ROUND(AVG(CASE WHEN f.orcamento_id IS NULL THEN 1.0 ELSE 0.0 END), 3)
                   FROM b_orcamento o JOIN b_usuario u ON u.id = o.vendedor_id
                   LEFT JOIN (SELECT DISTINCT orcamento_id FROM b_follow_up) f ON f.orcamento_id = o.id
                   WHERE o.tipo_orcamento_id = 1 AND o.dt_cancelou IS NULL
                   GROUP BY 1 HAVING COUNT(*) >= 100 ORDER BY 2 DESC""").df()
fig, ax = plt.subplots(figsize=(9, max(3, 0.32 * len(df))))
ax.barh(df.iloc[:, 0].astype(str), df.iloc[:, 1], color="#8c6d46")
ax.invert_yaxis()
ax.set_title('Follow-up ausente depende do vendedor')
ax.set_xlabel('taxa sem follow-up')
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


/tmp/ipykernel_107187/107204231.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Nota Técnica ACH-09**

- **Observado:** taxa observada de 16,42% na base analisada (métrica: `ACH-09`).
- **Por que importa:** O follow-up é a trilha de interação; se metade dos vendedores não registra, qualquer análise de esforço comercial compara disciplina de registro, não trabalho.
- **Ação (regra da silver):** `qtd_follow_ups` por orçamento (zero quando não há); a gold expõe a métrica por vendedor como indicador de aderência ao processo, não de esforço.

## 10. ACH-10 · Fora do escopo do BI: tipos não-venda e cancelados

In [13]:
df = con.execute("""SELECT t.nome AS tipo,
                  SUM(CASE WHEN o.dt_cancelou IS NULL THEN 1 ELSE 0 END) AS ativos,
                  SUM(CASE WHEN o.dt_cancelou IS NOT NULL THEN 1 ELSE 0 END) AS cancelados
           FROM b_orcamento o JOIN b_tipo_orcamento t ON t.id = o.tipo_orcamento_id
           GROUP BY 1 ORDER BY 2 DESC""").df()
display(df)
metrica = con.execute("""SELECT AVG(CASE WHEN tipo_orcamento_id <> 1 OR dt_cancelou IS NOT NULL THEN 1.0 ELSE 0.0 END) FROM b_orcamento""").fetchone()[0]
print("métrica ACH-10:", metrica)

,tipo,ativos,cancelados
0,VENDA,14180.0,213.0
1,ASSISTÊNCIA TÉCNICA,448.0,6.0
2,CORTESIA,222.0,2.0
3,GARANTIA,218.0,4.0


métrica ACH-10: 0.07277839534427516


**Nota Técnica ACH-10**

- **Observado:** taxa observada de 7,28% na base analisada (métrica: `ACH-10`).
- **Por que importa:** Assistência, garantia e cortesia são orçamentos no sistema e não são vendas; cancelados saem de todas as contas. Sem o filtro, volume e conversão vêm errados desde a primeira tela.
- **Ação (regra da silver):** Flag `fl_conta_bi` na silver (Venda e não cancelado); a gold só carrega linhas com a flag.

## 11. ACH-11 · Fase atual × trilha e data registrada × trilha

In [14]:
df = con.execute("""WITH ult AS (
             SELECT orcamento_id, fase_id, dt_entrada,
                    row_number() OVER (PARTITION BY orcamento_id ORDER BY dt_entrada DESC, id DESC) AS rn
             FROM b_orcamento_fase_hist)
           SELECT
             SUM(CASE WHEN o.fase_id = u.fase_id THEN 1 ELSE 0 END) AS fase_coerente,
             SUM(CASE WHEN o.fase_id <> u.fase_id THEN 1 ELSE 0 END) AS fase_divergente,
             SUM(CASE WHEN o.fase_id IN (6, 7) AND o.dt_finalizou = u.dt_entrada THEN 1 ELSE 0 END)
                 AS data_igual_trilha,
             SUM(CASE WHEN o.fase_id IN (6, 7) AND o.dt_finalizou <> u.dt_entrada THEN 1 ELSE 0 END)
                 AS data_diverge_trilha
           FROM b_orcamento o JOIN ult u ON u.orcamento_id = o.id AND u.rn = 1
           WHERE o.dt_cancelou IS NULL""").df()
display(df)
metrica = con.execute("""WITH ult AS (
             SELECT orcamento_id, fase_id,
                    row_number() OVER (PARTITION BY orcamento_id ORDER BY dt_entrada DESC, id DESC) AS rn
             FROM b_orcamento_fase_hist)
           SELECT AVG(CASE WHEN o.fase_id = u.fase_id THEN 1.0 ELSE 0.0 END)
           FROM b_orcamento o JOIN ult u ON u.orcamento_id = o.id AND u.rn = 1""").fetchone()[0]
print("métrica ACH-11:", metrica)

,fase_coerente,fase_divergente,data_igual_trilha,data_diverge_trilha
0,15068.0,0.0,13812.0,305.0


métrica ACH-11: 1.0


**Nota Técnica ACH-11**

- **Observado:** taxa observada de 100,00% na base analisada (métrica: `ACH-11`).
- **Por que importa:** A trilha de fases é a fonte da posição histórica do funil; se ela não bater com a fase atual do cabeçalho, nenhum 'quantos estavam abertos em agosto' é confiável.
- **Ação (regra da silver):** A trilha é a verdade para datas de fase e fechamento; o cabeçalho é a verdade para valores e vínculos. Divergências de data são as dos achados 01 e 02.

## Encerramento

Os achados acima entram no catálogo com id, regra e taxa observada. A silver implementa as regras e presta contas: para cada achado, quantas linhas foram afetadas, com o veredito que reprova a si mesmo quando a contagem não bate.